## 02 - Silver Transformation Pipeline
Cleans, type-casts, and enriches bronze data into the silver layer.

**Transformations:**
- Cast string columns to proper types (int, double, timestamp, boolean)
- Clean currency fields (remove `$` and `,`)
- Parse date/timestamp columns
- Enrich transactions with MCC category descriptions
- Join fraud labels to transactions

**Source:** `jrvs_databricks.bronze`  
**Target:** `jrvs_databricks.silver`

In [0]:
CATALOG = "jrvs_databricks"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

In [0]:
from pyspark.sql.functions import (
    col, trim, regexp_replace, to_timestamp, to_date, when, upper, lower,
    year, month, dayofweek, hour, date_format, lit, coalesce
)
from pyspark.sql.types import IntegerType, DoubleType, BooleanType, LongType

# Read bronze cards
cards_bronze = spark.table(f"{BRONZE}.cards")

# Clean and type-cast cards
silver_cards = (cards_bronze
    .withColumn("id", col("id").cast(IntegerType()))
    .withColumn("client_id", col("client_id").cast(IntegerType()))
    .withColumn("card_brand", trim(col("card_brand")))
    .withColumn("card_type", trim(col("card_type")))
    .withColumn("card_number", trim(col("card_number")))
    .withColumn("expires", trim(col("expires")))
    .withColumn("cvv", col("cvv").cast(IntegerType()))
    .withColumn("has_chip", when(upper(col("has_chip")) == "YES", True).otherwise(False))
    .withColumn("num_cards_issued", col("num_cards_issued").cast(IntegerType()))
    .withColumn("credit_limit", regexp_replace(col("credit_limit"), "[\\$,]", "").cast(DoubleType()))
    .withColumn("acct_open_date", to_date(col("acct_open_date"), "MM/yyyy"))
    .withColumn("year_pin_last_changed", col("year_pin_last_changed").cast(IntegerType()))
    .withColumn("card_on_dark_web",
        when(lower(col("card_on_dark_web")) == "yes", True)
        .when(lower(col("card_on_dark_web")) == "no", False)
        .otherwise(None).cast(BooleanType())
    )
)

silver_cards.write.mode("overwrite").saveAsTable(f"{SILVER}.cards")
print(f"Silver cards: {spark.table(f'{SILVER}.cards').count()} rows")
display(spark.table(f"{SILVER}.cards").limit(5))

Silver cards: 6146 rows


id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
4524,825,Visa,Debit,4344676511950444,12/2022,623,true,2,24295.0,2002-09-01,2008,false
2731,825,Visa,Debit,4956965974959986,12/2020,393,true,2,21968.0,2014-04-01,2014,false
3701,825,Visa,Debit,4582313478255491,02/2024,719,true,2,46414.0,2003-07-01,2004,false
42,825,Visa,Credit,4879494103069057,08/2024,693,false,1,12400.0,2003-01-01,2012,false
4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,true,1,28.0,2008-09-01,2009,false


In [0]:
# Read bronze users
users_bronze = spark.table(f"{BRONZE}.users_data")

# Clean and type-cast users
silver_users = (users_bronze
    .withColumn("id", col("id").cast(IntegerType()))
    .withColumn("current_age", col("current_age").cast(IntegerType()))
    .withColumn("retirement_age", col("retirement_age").cast(IntegerType()))
    .withColumn("birth_year", col("birth_year").cast(IntegerType()))
    .withColumn("birth_month", col("birth_month").cast(IntegerType()))
    .withColumn("gender", trim(col("gender")))
    .withColumn("address", trim(col("address")))
    .withColumn("latitude", col("latitude").cast(DoubleType()))
    .withColumn("longitude", col("longitude").cast(DoubleType()))
    .withColumn("per_capita_income", regexp_replace(col("per_capita_income"), "[\\$,]", "").cast(DoubleType()))
    .withColumn("yearly_income", regexp_replace(col("yearly_income"), "[\\$,]", "").cast(DoubleType()))
    .withColumn("total_debt", regexp_replace(col("total_debt"), "[\\$,]", "").cast(DoubleType()))
    .withColumn("credit_score", col("credit_score").cast(IntegerType()))
    .withColumn("num_credit_cards", col("num_credit_cards").cast(IntegerType()))
)

silver_users.write.mode("overwrite").saveAsTable(f"{SILVER}.users")
print(f"Silver users: {spark.table(f'{SILVER}.users').count()} rows")
display(spark.table(f"{SILVER}.users").limit(5))

Silver users: 2000 rows


id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,29278.0,59696.0,127613.0,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,37891.0,77254.0,191349.0,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,22681.0,33483.0,196.0,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,163145.0,249925.0,202328.0,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,53797.0,109687.0,183855.0,675,1


In [0]:
# Read bronze tables
txn_bronze = spark.table(f"{BRONZE}.transactions")
mcc_codes = spark.table(f"{BRONZE}.mcc_codes")
fraud_labels = spark.table(f"{BRONZE}.fraud_labels")

# Clean and type-cast transactions
txn_cleaned = (txn_bronze
    .withColumn("id", col("id").cast(LongType()))
    .withColumn("transaction_date", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("client_id", col("client_id").cast(IntegerType()))
    .withColumn("card_id", col("card_id").cast(IntegerType()))
    .withColumn("amount", regexp_replace(col("amount"), "[\\$,]", "").cast(DoubleType()))
    .withColumn("use_chip", trim(col("use_chip")))
    .withColumn("merchant_id", col("merchant_id").cast(LongType()))
    .withColumn("merchant_city", trim(col("merchant_city")))
    .withColumn("merchant_state", trim(col("merchant_state")))
    .withColumn("zip", trim(col("zip")))
    .withColumn("mcc", trim(col("mcc")))
    .withColumn("errors", trim(col("errors")))
    .drop("date")
)

# Enrich with MCC descriptions
txn_with_mcc = txn_cleaned.join(
    mcc_codes.select("mcc", "mcc_description"),
    on="mcc",
    how="left"
)

# Enrich with fraud labels (is_fraud is integer: 0 or 1)
silver_transactions = txn_with_mcc.join(
    fraud_labels.select(col("transaction_id"), col("is_fraud")),
    txn_with_mcc["id"] == fraud_labels["transaction_id"],
    how="left"
).drop("transaction_id")

# Add derived time columns useful for analysis
silver_transactions = (silver_transactions
    .withColumn("transaction_year", year("transaction_date"))
    .withColumn("transaction_month", month("transaction_date"))
    .withColumn("day_of_week", dayofweek("transaction_date"))  # 1=Sunday, 7=Saturday
    .withColumn("day_name", date_format("transaction_date", "EEEE"))
    .withColumn("hour_of_day", hour("transaction_date"))
    .withColumn("time_of_day",
        when((col("hour_of_day") >= 6) & (col("hour_of_day") < 12), "Morning")
        .when((col("hour_of_day") >= 12) & (col("hour_of_day") < 18), "Afternoon")
        .when((col("hour_of_day") >= 18) & (col("hour_of_day") < 22), "Evening")
        .otherwise("Night")
    )
    .withColumn("is_fraud", coalesce(col("is_fraud"), lit(0)))
)

silver_transactions.write.mode("overwrite").saveAsTable(f"{SILVER}.transactions")
print(f"Silver transactions: {spark.table(f'{SILVER}.transactions').count()} rows")
display(spark.table(f"{SILVER}.transactions").limit(5))

Silver transactions: 13305915 rows


mcc,id,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,transaction_date,mcc_description,is_fraud,transaction_year,transaction_month,day_of_week,day_name,hour_of_day,time_of_day
4829,7475440,706,4284,160.0,Swipe Transaction,27092,Boulder Creek,CA,95006.0,null,2010-01-01T02:39:00.000Z,Money Transfer,0,2010,1,6,Friday,2,Night
4784,7475490,585,5881,20.86,Online Transaction,41122,ONLINE,null,null,null,2010-01-01T04:14:00.000Z,Tolls and Bridge Fees,0,2010,1,6,Friday,4,Night
5411,7475634,571,5960,1.77,Swipe Transaction,49903,Martinsburg,WV,25401.0,null,2010-01-01T06:18:00.000Z,"Grocery Stores, Supermarkets",0,2010,1,6,Friday,6,Morning
5499,7475816,1382,2838,64.0,Swipe Transaction,59935,Dallas,TX,75254.0,null,2010-01-01T07:07:00.000Z,Miscellaneous Food Stores,0,2010,1,6,Friday,7,Morning
5541,7475873,437,3271,-90.0,Swipe Transaction,61195,Arnold,MD,21012.0,null,2010-01-01T07:20:00.000Z,Service Stations,0,2010,1,6,Friday,7,Morning


In [0]:
# Summary of silver tables
print("=" * 60)
print("SILVER TRANSFORMATION COMPLETE")
print("=" * 60)
for table in ["cards", "users", "transactions"]:
    count = spark.table(f"{SILVER}.{table}").count()
    cols = len(spark.table(f"{SILVER}.{table}").columns)
    print(f"  {SILVER}.{table}: {count:,} rows, {cols} columns")

# Fraud label stats
fraud_stats = spark.table(f"{SILVER}.transactions").groupBy("is_fraud").count().collect()
for row in fraud_stats:
    label = "Fraud" if row["is_fraud"] == 1 else "Legitimate"
    print(f"  {label} (is_fraud={row['is_fraud']}): {row['count']:,} transactions")
print("=" * 60)

SILVER TRANSFORMATION COMPLETE
  jrvs_databricks.silver.cards: 6,146 rows, 13 columns
  jrvs_databricks.silver.users: 2,000 rows, 14 columns
  jrvs_databricks.silver.transactions: 13,305,915 rows, 20 columns
  Fraud (is_fraud=1): 13,332 transactions
  Legitimate (is_fraud=0): 13,292,583 transactions
